# AVERAGE THE DATA IN THE SAME COORDINATES

In [31]:
import sys
print(sys.executable)

/usr/bin/python


In [32]:
#we import all they key libraries needed in this notebook
import numpy as np
import pandas as pd
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import matplotlib as matplotlib

### Required files:
We load the data and save it in a pandas dataframe.
- ***./csv/features/bact_data.csv***

In [33]:
nifh_df = pd.read_csv("./csv/features/bact_data.csv") 
print(nifh_df.columns)
nifh_df.info()

Index(['LATITUDE', 'LONGITUDE', 'Trichodesmium nifH Gene (x106 copies m-3)',
       'UCYN-A nifH Gene (x106 copies m-3)',
       'UCYN-B nifH Gene (x106 copies m-3)'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3641 entries, 0 to 3640
Data columns (total 5 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   LATITUDE                                   3641 non-null   float64
 1   LONGITUDE                                  3641 non-null   float64
 2   Trichodesmium nifH Gene (x106 copies m-3)  2700 non-null   float64
 3   UCYN-A nifH Gene (x106 copies m-3)         2910 non-null   float64
 4   UCYN-B nifH Gene (x106 copies m-3)         2313 non-null   float64
dtypes: float64(5)
memory usage: 142.4 KB


### Not all columns have enough data
Obviously we can tell that some columns like Gamma-A have not enough data, so we would rather not use it for our model. 
### There are duplicate coordinates
Also, for each of these types of bacteria there might be duplicate coordinates. In such cases we want to average the data on that coordinate. I have a suspicion that if we just group the main dataframe by coordinates and take mean it can lead to a lot of null values. I will test this theory.

**2 methods for averaging**:
1. simple average after group by
2. separate all columns average and join back together

### Simple average

In [34]:
#this takes an average in a simple way
nifh_simple_avg = nifh_df.groupby(by=['LATITUDE', 'LONGITUDE']).mean().reset_index()
print(nifh_simple_avg.columns)
nifh_simple_avg.info()

Index(['LATITUDE', 'LONGITUDE', 'Trichodesmium nifH Gene (x106 copies m-3)',
       'UCYN-A nifH Gene (x106 copies m-3)',
       'UCYN-B nifH Gene (x106 copies m-3)'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1029 entries, 0 to 1028
Data columns (total 5 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   LATITUDE                                   1029 non-null   float64
 1   LONGITUDE                                  1029 non-null   float64
 2   Trichodesmium nifH Gene (x106 copies m-3)  818 non-null    float64
 3   UCYN-A nifH Gene (x106 copies m-3)         851 non-null    float64
 4   UCYN-B nifH Gene (x106 copies m-3)         498 non-null    float64
dtypes: float64(5)
memory usage: 40.3 KB


In [35]:
nifh_simple_avg.describe()

,LATITUDE,LONGITUDE,Trichodesmium nifH Gene (x106 copies m-3),UCYN-A nifH Gene (x106 copies m-3),UCYN-B nifH Gene (x106 copies m-3)
count,1029.000000,1029.000000,8.180000e+02,8.510000e+02,498.000000
mean,21.181730,-6.353741,5.719386e+04,3.643482e+03,266.558928
std,25.741589,124.038964,1.316309e+06,8.122034e+04,3946.754100
min,-76.000000,-180.000000,0.000000e+00,0.000000e+00,0.000000
25%,10.000000,-126.000000,1.853400e+00,1.576421e-01,0.000000
50%,25.000000,-32.000000,3.895250e+01,5.010000e+00,0.434250
75%,35.000000,135.000000,2.513493e+02,7.983355e+01,9.305100
max,120.000000,187.000000,3.630781e+07,2.360150e+06,87605.116104


In [36]:
nifh_simple_avg.to_csv("./csv/features/bact_data_avg2.csv", index=False)

### Complicated average
We store the list of columns that we intend to learn from and create the model. I separated the data columns and the coordinates.

In [37]:
data_cols = ['Trichodesmium nifH Gene (x106 copies m-3)','UCYN-A nifH Gene (x106 copies m-3)','UCYN-B nifH Gene (x106 copies m-3)']
cor_cols = ['LATITUDE', 'LONGITUDE']

In [38]:
frames = []

for col in data_cols:
    #we only take a look at specific column from the dataset
    selected_data = nifh_df[cor_cols+[col]]

    #we only want to average the not na values in order not to produce nulls
    notna_mask = selected_data[col].notna()
    selected_data_notna = selected_data[notna_mask]
    print("{0}: not null values: {1}, shape: {2}".format(col, notna_mask.sum(), selected_data_notna.shape))#we can verify that we have only not null values

    #now we can average based on coordinates
    selected_data_avg = selected_data_notna.groupby(by=['LATITUDE', 'LONGITUDE']).mean()
    print("{0}: shape after averaging: {1}\n".format(col, selected_data_avg.shape))

    #we add the modified dataframe to the list
    frames.append(selected_data_avg)


Trichodesmium nifH Gene (x106 copies m-3): not null values: 2700, shape: (2700, 3)
Trichodesmium nifH Gene (x106 copies m-3): shape after averaging: (818, 1)

UCYN-A nifH Gene (x106 copies m-3): not null values: 2910, shape: (2910, 3)
UCYN-A nifH Gene (x106 copies m-3): shape after averaging: (851, 1)

UCYN-B nifH Gene (x106 copies m-3): not null values: 2313, shape: (2313, 3)
UCYN-B nifH Gene (x106 copies m-3): shape after averaging: (498, 1)



In [39]:
#concatenate based on index(coordinates)
nifh_avg = pd.concat(frames, axis=1)

nifh_avg.head()

Trichodesmium nifH Gene (x106 copies m-3)  \
LATITUDE LONGITUDE                                              
-76.0    34.0                                        0.329000   
         35.0                                        0.057000   
-75.0    35.0                                        0.103500   
-74.0    35.0                                        1.635286   
         36.0                                        7.797333   

                    UCYN-A nifH Gene (x106 copies m-3)  \
LATITUDE LONGITUDE                                       
-76.0    34.0                                      NaN   
         35.0                                      NaN   
-75.0    35.0                                      NaN   
-74.0    35.0                                      NaN   
         36.0                                      NaN   

                    UCYN-B nifH Gene (x106 copies m-3)  
LATITUDE LONGITUDE                                      
-76.0    34.0                                      NaN  
         35.0                                      NaN  
-75.0    35.0                                      NaN  
-74.0    35.0                                      NaN  
         36.0                                      NaN

In [40]:
nifh_avg.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 1029 entries, (-76.0, 34.0) to (30.0, -89.0)
Data columns (total 3 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   Trichodesmium nifH Gene (x106 copies m-3)  818 non-null    float64
 1   UCYN-A nifH Gene (x106 copies m-3)         851 non-null    float64
 2   UCYN-B nifH Gene (x106 copies m-3)         498 non-null    float64
dtypes: float64(3)
memory usage: 42.5 KB


In [41]:
nifh_avg.describe()

,Trichodesmium nifH Gene (x106 copies m-3),UCYN-A nifH Gene (x106 copies m-3),UCYN-B nifH Gene (x106 copies m-3)
count,8.180000e+02,8.510000e+02,498.000000
mean,5.719386e+04,3.643482e+03,266.558928
std,1.316309e+06,8.122034e+04,3946.754100
min,0.000000e+00,0.000000e+00,0.000000
25%,1.853400e+00,1.576421e-01,0.000000
50%,3.895250e+01,5.010000e+00,0.434250
75%,2.513493e+02,7.983355e+01,9.305100
max,3.630781e+07,2.360150e+06,87605.116104


As we can see the results are exactly the same so a simpler one line method can be used to produce the same result